# C0003R03 PARK25 LC - SCAN2 Mass Correction

This notebook **does not generate a new Excel table**.

It is used only to correct the mass columns in the existing SCAN2 result file generated by
`C0003R03_PARK25_LC_1D_SCAN.ipynb`:

`out/C0003/C0003R03_PARK25_LC_1D_SCAN.xlsx`

Only the following columns in sheet `Geometry scan_stage2` are overwritten:

- `m_plate_kg`
- `m_coolant_kg`
- `mtotal`

All thermal, flow, pump-power, and mission results remain unchanged.

The corrected cold-plate width is determined by two geometric constraints:

\[
W_{battery}=16\times18=288\;\mathrm{mm}
\]

\[
S_{channel}\ge1\;\mathrm{mm}
\]

For each SCAN2 row:

\[
W_{plate,spacing}=N_{channel}W_{channel}+(N_{channel}+1)S_{min}
\]

\[
W_{plate}=\max\left(W_{battery},\;W_{plate,spacing}\right)
\]

The corrected mass is then calculated by directly calling
`BTMS_model.cal_btms_mass()`.


In [19]:
# ============================================================
# Cell 0 - Import packages, load BTMS_model, and locate SCAN2 Excel
# ============================================================

import os
import sys
import importlib

import CoolProp.CoolProp as CP
from openpyxl import load_workbook

# Keep the same project convention as C0003R03_PARK25_LC_1D_SCAN.ipynb.
PROJECT_ROOT = os.path.abspath('../..')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

try:
    import lib.BTMS_model as BTMS_model
except ModuleNotFoundError:
    import BTMS_model

BTMS_model = importlib.reload(BTMS_model)

CASE_ID = 'C0003'
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'out', CASE_ID)

# IMPORTANT:
# This is the Excel file produced by the original SCAN notebook.
# This notebook updates that file in place and does not create a new result workbook.
SCAN_XLSX_PATH = os.path.join(
    OUTPUT_DIR,
    'C0003R03_PARK25_LC_1D_SCAN.xlsx',
)
SCAN2_SHEET = 'Geometry scan_stage2'

print(f'Project root: {PROJECT_ROOT}')
print(f'BTMS_model loaded from: {BTMS_model.__file__}')
print(f'SCAN2 Excel to update: {SCAN_XLSX_PATH}')
print(f'Sheet to update: {SCAN2_SHEET}')


Project root: d:\eBATS
BTMS_model loaded from: d:\eBATS\lib\BTMS_model.py
SCAN2 Excel to update: d:\eBATS\out\C0003\C0003R03_PARK25_LC_1D_SCAN.xlsx
Sheet to update: Geometry scan_stage2


In [20]:
# ============================================================
# Cell 1 - Mass-correction settings
# ============================================================

# Same SCAN2 inlet-water condition used by the original notebook.
T_water = 25.0                  # degC
p_water = 101325.0              # Pa

# Battery-array transverse geometry.
N_c = 16
D_bat = 18.0e-3                 # m
W_battery = N_c * D_bat         # m = 288 mm

# Minimum clear spacing between adjacent channel walls / outer boundaries.
S_min = 1.0e-3                  # m = 1 mm

# Same plate/material settings as the original model.
H_bottom = 2.0e-3               # m
rho_plate = 2719.0              # kg/m3, aluminium
m_pump = 0.0                    # kg
m_pipe = 0.0                    # kg

print(f'W_battery = {W_battery * 1e3:.1f} mm')
print(f'S_min = {S_min * 1e3:.1f} mm')
print(f'H_bottom = {H_bottom * 1e3:.1f} mm')


W_battery = 288.0 mm
S_min = 1.0 mm
H_bottom = 2.0 mm


In [21]:
# ============================================================
# Cell 2 - Water density at the SCAN2 inlet condition
# ============================================================

rho_water = CP.PropsSI(
    'D',
    'T', T_water + 273.15,
    'P', p_water,
    'Water',
)

print(f'rho_water = {rho_water:.6f} kg/m3')


rho_water = 997.047637 kg/m3


In [22]:
# ============================================================
# Cell 3 - Correct mass calculation for one existing SCAN2 row
# ============================================================

def calculate_corrected_mass(
    W_channel,
    H_channel,
    L_channel,
    num_parallel_channels,
):
    """
    Recalculate BTMS mass for one existing SCAN2 geometry row.

    The cold-plate width is the minimum value satisfying:
      1) W_plate >= 288 mm battery-array width
      2) S_channel >= 1 mm

    Mass itself is calculated only by BTMS_model.cal_btms_mass().
    """

    num_parallel_channels = int(num_parallel_channels)
    num_spacing_intervals = num_parallel_channels + 1

    # Minimum plate width required when every spacing interval is exactly 1 mm.
    W_plate_spacing_limit = (
        num_parallel_channels * W_channel
        + num_spacing_intervals * S_min
    )

    # Minimum feasible plate width under both constraints.
    W_plate = max(
        W_battery,
        W_plate_spacing_limit,
    )

    # Derived actual clear spacing, used only for geometry validation.
    S_channel = (
        W_plate - num_parallel_channels * W_channel
    ) / num_spacing_intervals

    tol = 1e-12

    if W_plate < W_battery - tol:
        raise RuntimeError(
            'Cold-plate width is smaller than the 288 mm battery-array width.'
        )

    if S_channel < S_min - tol:
        raise RuntimeError(
            'Calculated channel spacing is below the 1 mm minimum: '
            f'{S_channel * 1e3:.6f} mm.'
        )

    A_cool_cs = W_channel * H_channel

    mass_args = {
        'fluid_cool': 'Water',
        'rho_cool': rho_water,
        'rho_plate': rho_plate,
        'L_channel': L_channel,
        'W_plate': W_plate,
        'A_cool_cs': A_cool_cs,
        'H_channel': H_channel,
        'H_bottom': H_bottom,
        'num_channel': num_parallel_channels,
        'm_pump': m_pump,
        'm_pipe': m_pipe,
        'return_components': True,
    }

    mass_results = BTMS_model.cal_btms_mass(mass_args)

    return {
        'm_plate_kg': float(mass_results['m_plate_kg']),
        'm_coolant_kg': float(mass_results['m_coolant_kg']),
        'mtotal': float(mass_results['m_BTMS_kg']),
        # Only for console verification; these values are NOT added to the Excel sheet.
        'W_plate_mm': float(W_plate * 1e3),
        'S_channel_mm': float(S_channel * 1e3),
    }


In [23]:
# ============================================================
# Cell 4 - Overwrite only the wrong mass columns in the existing SCAN2 sheet
# ============================================================

if not os.path.exists(SCAN_XLSX_PATH):
    raise FileNotFoundError(
        'The SCAN2 Excel file does not exist. Run '
        'C0003R03_PARK25_LC_1D_SCAN.ipynb first so that this file is created:\n'
        f'{SCAN_XLSX_PATH}'
    )

wb = load_workbook(SCAN_XLSX_PATH)

if SCAN2_SHEET not in wb.sheetnames:
    raise KeyError(
        f'Sheet {SCAN2_SHEET!r} was not found in {SCAN_XLSX_PATH}. '
        f'Available sheets: {wb.sheetnames}'
    )

ws = wb[SCAN2_SHEET]

# Build a header -> column-number map from the existing table.
header_map = {
    cell.value: cell.column
    for cell in ws[1]
    if cell.value is not None
}

required_columns = [
    'W_channel_mm',
    'H_channel_mm',
    'L_channel_mm',
    'num_parallel_channels',
    'm_plate_kg',
    'm_coolant_kg',
    'mtotal',
]

missing_columns = [
    name for name in required_columns
    if name not in header_map
]

if missing_columns:
    raise KeyError(
        'Required columns are missing from Geometry scan_stage2: '
        + ', '.join(missing_columns)
    )

updated_rows = 0
plate_widths_mm = []
channel_spacings_mm = []

for row_idx in range(2, ws.max_row + 1):
    W_mm = ws.cell(row_idx, header_map['W_channel_mm']).value
    H_mm = ws.cell(row_idx, header_map['H_channel_mm']).value
    L_mm = ws.cell(row_idx, header_map['L_channel_mm']).value
    N_channel = ws.cell(row_idx, header_map['num_parallel_channels']).value

    # Ignore completely empty rows below the table, if any.
    if all(value is None for value in (W_mm, H_mm, L_mm, N_channel)):
        continue

    if any(value is None for value in (W_mm, H_mm, L_mm, N_channel)):
        raise ValueError(
            f'Incomplete SCAN2 geometry data at Excel row {row_idx}.'
        )

    result = calculate_corrected_mass(
        W_channel=float(W_mm) * 1e-3,
        H_channel=float(H_mm) * 1e-3,
        L_channel=float(L_mm) * 1e-3,
        num_parallel_channels=int(N_channel),
    )

    # ---------------------------------------------------------
    # IMPORTANT: only these three cells are overwritten.
    # All other SCAN2 results remain exactly as generated by the original notebook.
    # ---------------------------------------------------------
    ws.cell(row_idx, header_map['m_plate_kg']).value = result['m_plate_kg']
    ws.cell(row_idx, header_map['m_coolant_kg']).value = result['m_coolant_kg']
    ws.cell(row_idx, header_map['mtotal']).value = result['mtotal']

    updated_rows += 1
    plate_widths_mm.append(result['W_plate_mm'])
    channel_spacings_mm.append(result['S_channel_mm'])

# Save back to the SAME Excel file.
# Existing sheet structure, formatting, thermal results, pump results, etc. are preserved.
wb.save(SCAN_XLSX_PATH)

print('SCAN2 mass correction completed.')
print(f'Updated rows: {updated_rows}')
print('Updated columns only: m_plate_kg, m_coolant_kg, mtotal')
print(f'Overwritten Excel file: {SCAN_XLSX_PATH}')
print(
    'Cold-plate-width range used in corrected mass calculation: '
    f'{min(plate_widths_mm):.3f} to {max(plate_widths_mm):.3f} mm'
)
print(
    'Channel-spacing range used in corrected mass calculation: '
    f'{min(channel_spacings_mm):.3f} to {max(channel_spacings_mm):.3f} mm'
)


SCAN2 mass correction completed.
Updated rows: 100
Updated columns only: m_plate_kg, m_coolant_kg, mtotal
Overwritten Excel file: d:\eBATS\out\C0003\C0003R03_PARK25_LC_1D_SCAN.xlsx
Cold-plate-width range used in corrected mass calculation: 288.000 to 353.000 mm
Channel-spacing range used in corrected mass calculation: 1.000 to 2.909 mm
